Рекурсивные функции полезны при работе с задачами, где решение строится на основе более простых подзадач того же типа.  
- Задать базовый случай — что функция должна вернуть при простейшем значении аргумента

- Задать рекурсивное правило — как вычислить результат на основе значения функции

In [ ]:
def factorial(n):
    if n == 0:
        return 1
    return factorial(n - 1) * n

print(factorial(5))

Функция начинает разворачиваться в цепочку вызовов.   

Python не завершает текущую функцию, а «замораживает» её и кладет в специальную память - **стек вызовов (call stack)**.  

Ограничение: 1000 вложенных вызовов.

- **factorial(5)** видит, что n != 0, и возвращает factorial(4) * 5  

- **factorial(4)** видит, что n != 0, и возвращает factorial(3) * 4  

- **factorial(3)** видит, что n != 0, и возвращает factorial(2) * 3  

- **factorial(2)** видит, что n != 0, и возвращает factorial(1) * 2  

- **factorial(1)** видит, что n != 0, и возвращает factorial(0) * 1  

Обратный ход (вычисление результата)  

- factorial(0) вернул 1.  

- factorial(1) считает: 1 * 1 = 1.  

- factorial(2) считает: 1 * 2 = 2.  

- factorial(3) считает: 2 * 3 = 6.  

- factorial(4) считает: 6 * 4 = 24.  

- factorial(5) считает: 24 * 5 = **120**

>p.s. math написан на языке C, работает эффективнее всего

In [ ]:
import math
print(math.factorial(2000))  # Написано на языке C, работает эффективнее всего

*Декларативный стиль* версия функции с рекурсией `def func(): return func()` (код более выразительным и компактным, `по заданной Математической формуле`)

*Императивный стиль* версия функции с циклом `for i in range...` (расписываем подробно действия)


>Императивный стиль фокусируется на процессе: он отвечает на вопрос «как это сделать?», тогда как декларативный — на «что именно нужно получить?».  

#### Рекурсивные алгоритмы могут быть медленными  
Пример *последовательность Фибоначчи*
- Первые два числа равны 1
- Каждое последующее — сумма двух предыдущих.

In [ ]:
from timeit import timeit

# декларативно, рекурсивное дерево 
def fib(n):
    if n in (0, 1):
        return 1
    return fib(n - 1) + fib(n - 2)

print(f"Среднее время вычисления: "
     f"{round(timeit('fib(35)', number=10, globals=globals()) / 10, 3)} с.")
# Среднее время вычисления: 1.615 с. (29860703 вызовов)



# императивно
def fib(n):
    f_1, f = 1, 1
    for _ in range(n - 1):
        f_1, f = f, f_1 + f
    return f

print(f"Среднее время вычисления: "
     f"{round(timeit('fib(35)', number=10, globals=globals()) / 10, 3)} с.")
# Среднее время вычисления: 0.0 с.


Решение для деклоративного подхода — запоминать уже вычисленные значения, чтобы не считать их повторно. Такой приём называется `кешированием` или `мемоизацией`.
Один из способов — сохранять промежуточные результаты в словаре:

In [ ]:
from timeit import timeit
from sys import setrecursionlimit # увеличить лимит вложенности

setrecursionlimit(2000)
cache = {0: 1, 1: 1}

def fib(n):
    if n not in cache:
        cache[n] = fib(n - 1) + fib(n - 2)
    return cache[n]

execution_time = timeit('fib(1000)', number=10, globals=globals()) / 10
print(f"Среднее время вычисления: {execution_time:.6f} с.")

# 35-е число Фибоначчи равно: 14930352.
# Количество вызовов рекурсивной функции равно: 69.
# Среднее время вычисления: 1e-06 с.


#### Можно поручить кеширование самому интерпретатору  
При `maxsize=None`: Кэш становится бесконечным. Операция удаления старых записей отключается, поэтому такой кэш работает чуть-чуть быстрее. Рекомендуется устанавливать размер кэша равным степени двойки для оптимальной производительности процессора. (128, 256, 1024)

In [ ]:
from timeit import timeit
from functools import lru_cache

@lru_cache(maxsize=1024)
def fib(n):
    if n in (0, 1):
        return 1
    return fib(n - 1) + fib(n - 2)

print(fib.cache_info()) 
# CacheInfo(hits=197, misses=101, maxsize=1024, currsize=101)

Главный вывод: Размер `maxsize` всегда должен быть не меньше, чем `количество уникальных аргументов`, которые функция должна помнить одновременно для эффективной работы, иначе произойдет *Потеря эффекта мемоизации*.

>Вместо 100 пересчетов fib(100), посчитался первый, а остальные 99 просто взяли данные с lru_cache

### Декораторы  
Декоратор — это специальная конструкция, которая оборачивает функцию и может изменить её поведение до, после или вместо обычного вызова. (например, добавить логирование, измерение времени выполнения или кеширование результатов)

>Декоратор возвращает новую функцию, которая сначала выполняет дополнительную логику, а затем вызывает оригинальную.

In [ ]:
def func():
    return "Se"

def decorator(old_func):
    def new_func():
        return old_func()
    return new_func

print(func)
func = decorator(func)
print(func)

In [ ]:
from functools import lru_cache

def count_calls(f):
    def decorated(*args, **kwargs):
        # nonlocal total     # Инкапсуляция
        decorated.total += 1 # Открытый интерфейс
        return f(*args, **kwargs)
    
    decorated.total = 0
    return decorated

@count_calls
@lru_cache(maxsize=32)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

result = fib(35)
print(f"Результат: {result}")       # Результат: 9227465
print(f"Всего вызовов: {fib.total}")  # Всего вызовов: 69 (вместо 3213213321 без кеша)


Результат: 9227465
Всего вызовов: 69


### Генераторы  
Экономят память и ускоряют работу

In [ ]:
squares = (i ** 2 for i in range(10))
print(squares)

# Вывод программы:
# <generator object <genexpr> at 0x000001C225EFC9E0>

Чтобы получить значение, вызывается метод `__next__()` (обычно неявно — например, в цикле `for`).  

Чтобы написать свою собственную функцию-генератор, вместо оператора return нужно использовать `yield`:
- приостанавливает выполнение функции;  
- возвращает текущее значение;  
- позволяет продолжить выполнение с того места, где функция остановилась.  

Генераторная версия функции Фибоначчи:

In [ ]:
def fib(n):
    n_1, n_2 = 1, 1
    for _ in range(n):
        yield n_1
        n_1, n_2 = n_2, n_1 + n_2 # двигаем окно вправо


print(", ".join(str(x) for x in fib(10)))

# Вывод программы:
# 1, 1, 2, 3, 5, 8, 13, 21, 34, 55

1, 1, 2, 3, 5, 8, 13, 21, 34, 55


Если вызвать `fib(10000000)`, программа не будет создавать в оперативной памяти список из миллиона элементов. Она будет держать в памяти одновременно только два числа (n_1 и n_2)

**Обычная функция: O(n)**  
1. Создается фрейм (объект кадра выполнения/стека), где хранятся локальные переменные. Как только функция доходит до `return`, фрейм уничтожается из памяти навсегда.  
2. Есть доступ по индексу и можно пройтись повторно

**Генератор: O(1)**  

1. `yield` помечает функцию специальным флагом `CO_GENERATOR`.  
2. При вызове `fib(10)` Python возвращает **объект генератора** (`PyGenObject`).
3. Объект генератора внутри себя хранит ссылку на фрейм выполнения функции (`gi_frame`).
4. Когда вы вызываете `next(generator)`, Python заходит в этот фрейм и выполняет код до первого `yield`.
5. Как только код доходит до `yield`, Python берет значение, возвращает его наружу, но **не уничтожает фрейм**.
6. Интерпретатор сохраняет состояние всех локальных переменных и **указатель на следующую инструкцию байт-кода** (`f_lasti`). Фрейм просто "замораживается" в памяти.
7. При следующем вызове `next()` Python не начинает функцию заново, а смотрит на сохраненный `f_lasti` и продолжает выполнять байт-код _ровно с той точки, где остановился_.

Когда цикл или код внутри генератора завершается (или срабатывает явный `return`), выкидывается исключение `StopIteration`. Для циклов `for` в Python это сигнал, что элементы закончились, и они плавно завершают работу.